In [44]:
import pandas as pd
import numpy as np
import os

os.chdir('/Users/liujingxin/Documents/Quant_Project')

df = pd.read_csv('data/raw_monthly_data.csv', parse_dates=['trade_date'])
adj = pd.read_csv('data/adj_factor.csv', parse_dates=['trade_date'])

# 合并复权因子，算复权收盘价（已处理送转股和分红除息）
df = df.merge(adj[['ts_code', 'trade_date', 'adj_factor']],
              on=['ts_code', 'trade_date'], how='left')
print('缺复权因子的行数：', df['adj_factor'].isna().sum())
df['adj_close'] = df['close'] * df['adj_factor']

# 给每个月编号，用来确认"下一行"真的是下一个月
months = sorted(df['trade_date'].unique())
df['m_idx'] = df['trade_date'].map({d: i for i, d in enumerate(months)})
df = df.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)

# 下月收益 = 下月复权价 / 本月复权价 - 1
# 只有下一行恰好是下一个月才算，缺月的情况记为 NaN，不再算出跨多月收益
g = df.groupby('ts_code')
next_price = g['adj_close'].shift(-1)
next_month = g['m_idx'].shift(-1)
df['forward_return'] = np.where(next_month == df['m_idx'] + 1,
                                next_price / df['adj_close'] - 1, np.nan)

print(f'原始数据：{len(df):,} 行，{df["ts_code"].nunique():,} 只股票')
print(df['forward_return'].describe())
print('单月涨跌幅超过 ±50% 的行数：', (df['forward_return'].abs() > 0.5).sum())

缺复权因子的行数： 67
原始数据：349,215 行，5,675 只股票
count    342751.000000
mean          0.012418
std           0.144356
min          -0.868972
25%          -0.066502
50%          -0.002191
75%           0.068505
max           4.563492
Name: forward_return, dtype: float64
单月涨跌幅超过 ±50% 的行数： 3287


In [45]:
# 记录每一步过滤掉了多少行，方便你理解数据变化
print(f'清洗前：{len(df):,} 行')

# 第一步：删除 pe_ttm 或 pb 缺失的行
# 这两个是我们构建因子的核心数据，缺了就没法用
# dropna() 是"删除含有缺失值的行"的函数
# subset 指定只看这两列
df = df.dropna(subset=['pe_ttm', 'pb'])
print(f'删除 PE/PB 缺失后：{len(df):,} 行')

# 第二步：删除 pe_ttm 或 pb 为负数或零的行
# PE 负数代表亏损公司，EP 因子对这类公司没有意义
# PB 负数代表资不抵债，同样排除
# 用布尔索引筛选：只保留 pe_ttm > 0 且 pb > 0 的行
df = df[(df['pe_ttm'] > 0) & (df['pb'] > 0)]
print(f'删除 PE/PB 负值后：{len(df):,} 行')

# 第三步：删除行业缺失的行
# 我们要做行业中性化，没有行业分类的股票没法处理
df = df.dropna(subset=['industry'])
print(f'删除行业缺失后：{len(df):,} 行')

# 第四步：过滤新股
# 新上市不足 6 个月的股票数据不稳定，排除掉
# 思路：用上市日期和当前交易日期对比，如果差距小于 180 天就过滤
# 先读取股票基本信息（里面有上市日期）
stock_info = pd.read_csv('data/stock_industry.csv', parse_dates=['list_date'])

# 只保留股票代码和上市日期两列
stock_info = stock_info[['ts_code', 'list_date']]

# 把上市日期合并进 df
df = df.merge(stock_info, on='ts_code', how='left')

# 计算上市天数：交易日期 - 上市日期，取天数
df['days_listed'] = (df['trade_date'] - df['list_date']).dt.days

# 只保留上市超过 180 天的股票
df = df[df['days_listed'] >= 180]
print(f'过滤新股（上市<180天）后：{len(df):,} 行')

# 删除 list_date 和 days_listed 这两列，后面用不到了
df = df.drop(columns=['list_date', 'days_listed'])

print(f'\n清洗完成，剩余 {len(df):,} 行，{df["ts_code"].nunique():,} 只股票')

清洗前：349,215 行
删除 PE/PB 缺失后：275,097 行
删除 PE/PB 负值后：275,097 行
删除行业缺失后：271,971 行
过滤新股（上市<180天）后：260,630 行

清洗完成，剩余 260,630 行，5,325 只股票


In [46]:
# 每个月内，把 pe_ttm 和 pb 的极端值截到 1% 和 99% 分位数
for col in ['pe_ttm', 'pb']:
    df[col] = df.groupby('trade_date')[col].transform(
        lambda s: s.clip(s.quantile(0.01), s.quantile(0.99)))

print('Winsorize 完成 ✓')
print(f'pe_ttm 处理后范围：{df["pe_ttm"].min():.2f} 到 {df["pe_ttm"].max():.2f}')
print(f'pb 处理后范围：{df["pb"].min():.2f} 到 {df["pb"].max():.2f}')

Winsorize 完成 ✓
pe_ttm 处理后范围：4.03 到 1627.68
pb 处理后范围：0.48 到 26.52


In [47]:
# EP 因子 = 1 / PE_ttm
# 含义：每一块钱市值能对应多少盈利
# EP 越高 → 股票越便宜（盈利相对于股价高）
df['EP'] = 1 / df['pe_ttm']

# BP 因子 = 1 / PB
# 含义：每一块钱市值能对应多少账面净资产
# BP 越高 → 股票越便宜（账面价值相对于股价高）
df['BP'] = 1 / df['pb']

print('原始因子构建完成 ✓')
print(f'\nEP 统计：')
print(df['EP'].describe())
print(f'\nBP 统计：')
print(df['BP'].describe())

原始因子构建完成 ✓

EP 统计：
count    260630.000000
mean          0.038746
std           0.035037
min           0.000614
25%           0.015466
50%           0.029252
75%           0.050238
max           0.248040
Name: EP, dtype: float64

BP 统计：
count    260630.000000
mean          0.487974
std           0.322839
min           0.037711
25%           0.260554
50%           0.410593
75%           0.627392
max           2.079097
Name: BP, dtype: float64


In [48]:
# 行业中性化：每个月、每个行业内做 z-score
for f in ['EP', 'BP']:
    g = df.groupby(['trade_date', 'industry'])[f]
    std = g.transform('std').replace(0, np.nan)
    df[f'{f}_neutralized'] = (df[f] - g.transform('mean')) / std


In [49]:
# 市值中性化：每个月对 log(市值) 回归取残差，按索引写回，不会错位
df['log_mv'] = np.log(df['total_mv'])
for f in ['EP', 'BP']:
    out = pd.Series(np.nan, index=df.index)
    for date, idx in df.groupby('trade_date').groups.items():
        sub = df.loc[idx, [f'{f}_neutralized', 'log_mv']].dropna()
        if len(sub) < 10:
            continue
        X = np.column_stack([np.ones(len(sub)), sub['log_mv'].values])
        y = sub[f'{f}_neutralized'].values
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        out.loc[sub.index] = y - X @ beta
    df[f'{f}_final'] = out

In [50]:
# forward_return 已在 Cell 1 算好（复权价、清洗前），这里只做检查
n_old_style = len(df) - df['ts_code'].nunique()
print('可用行数：', df['forward_return'].notna().sum(), '| 旧算法会得到的行数：', n_old_style)
print(df['forward_return'].describe())

可用行数： 256461 | 旧算法会得到的行数： 255305
count    256461.000000
mean          0.012160
std           0.136123
min          -0.660302
25%          -0.063808
50%          -0.001559
75%           0.066919
max           4.256410
Name: forward_return, dtype: float64


In [51]:
# 保存最终的因子数据
# 只保留我们需要的列
assert 'monthly_return' not in df.columns, '旧的未复权收益单元格被运行过了'

cols_to_save = [
    'ts_code', 'trade_date', 'industry', 'total_mv',
    'EP', 'BP',                          # 原始因子值
    'EP_neutralized', 'BP_neutralized',  # 行业中性化后
    'EP_final', 'BP_final',              # 行业+市值中性化后（最终用这个）
    'forward_return'                      # 下个月收益率（预测目标）
]

factor_data = df[cols_to_save].copy()

# 删除 forward_return 为空的行
# 最后一个月的数据没有"下个月收益率"，自然是 NaN，删掉
factor_data = factor_data.dropna(subset=['forward_return'])

factor_data.to_csv('data/factor_data.csv', index=False)

print(f'因子数据已保存到 data/factor_data.csv')
print(f'总行数：{len(factor_data):,}')
print(f'时间范围：{factor_data["trade_date"].min()} 到 {factor_data["trade_date"].max()}')
print(f'\n预览：')
factor_data.head(10)

因子数据已保存到 data/factor_data.csv
总行数：256,461
时间范围：2020-01-23 00:00:00 到 2025-11-28 00:00:00

预览：


,ts_code,trade_date,industry,total_mv,EP,BP,EP_neutralized,BP_neutralized,EP_final,BP_final,forward_return
0,000001.SZ,2020-01-23,银行,3.015680e+07,0.093495,0.905387,-1.178178,-0.865714,-1.696011,-0.542067,-0.066924
1,000001.SZ,2020-02-28,银行,2.813858e+07,0.100200,0.970309,-1.089199,-0.803693,-1.549154,-0.420904,-0.117241
2,000001.SZ,2020-03-31,银行,2.483958e+07,0.113509,1.099143,-0.995878,-0.654201,-1.554694,-0.369827,0.088281
3,000001.SZ,2020-04-30,银行,2.703244e+07,0.108378,1.044714,-1.140160,-0.934224,-1.621241,-0.601894,-0.050691
4,000001.SZ,2020-05-29,银行,2.522769e+07,0.116131,1.300898,-1.013357,-0.142142,-1.447894,0.192428,-0.015385
5,000001.SZ,2020-06-30,银行,2.483958e+07,0.117945,1.119946,-0.936518,-0.679345,-1.284079,-0.290491,0.042188
6,000001.SZ,2020-07-31,银行,2.588749e+07,0.113171,1.074576,-0.886258,-0.581121,-1.214012,-0.179895,0.130435
7,000001.SZ,2020-08-31,银行,2.926412e+07,0.090452,0.961723,-1.278521,-0.781297,-1.658737,-0.381256,0.005968
8,000001.SZ,2020-09-30,银行,2.943878e+07,0.089915,0.956023,-1.442748,-0.967207,-1.798478,-0.564960,0.170073
9,000001.SZ,2020-10-30,银行,3.444550e+07,0.078304,0.838504,-1.725670,-1.393465,-2.111007,-0.990207,0.112113


In [52]:
cols = ['ts_code', 'trade_date', 'adj_factor', 'close', 'forward_return']
print(df.nlargest(10, 'forward_return')[cols].to_string())
print(df.nsmallest(5, 'forward_return')[cols].to_string())

          ts_code trade_date  adj_factor  close  forward_return
241606  688068.SH 2021-03-31      1.0045  42.90        4.256410
214237  603366.SH 2024-10-31      2.5421   4.41        3.938776
270109  920799.BJ 2024-09-30      3.1212  17.78        3.646232
107966  300641.SZ 2024-03-29      1.7915   4.31        3.617169
269900  920770.BJ 2024-09-30      1.0676   7.07        3.605375
264868  920204.BJ 2024-09-30      3.0169   8.92        3.168161
271720  920971.BJ 2024-09-30      1.8764  10.84        2.948339
264273  920139.BJ 2024-09-30      1.0061   9.81        2.786952
200935  603038.SH 2024-09-30      3.8395   8.99        2.634038
168483  600611.SH 2024-06-28    710.9315   2.79        2.559140
          ts_code trade_date  adj_factor  close  forward_return
208678  603222.SH 2020-11-30      2.0303  39.80       -0.660302
220904  603655.SH 2020-11-30      1.0128  62.65       -0.655866
102990  300551.SZ 2022-03-31      4.0805  17.14       -0.568261
228037  603879.SH 2020-05-29      1.4112